# Section 50D — Advanced Git

**Data Science and Machine Learning Course**

---

## Learning Objectives

By the end of this section you will be able to:
- Save and restore work-in-progress with `git stash`
- Mark releases with `git tag`
- Rewrite branch history with `git rebase`
- Apply individual commits with `git cherry-pick`
- Exclude files with `.gitignore`
- Apply Git best practices specific to data science projects
- Use `git bisect` to locate the commit that introduced a bug
- Create shortcuts with Git aliases

---

## 1. Stashing Changes — `git stash`

`git stash` temporarily shelves your uncommitted changes so you can switch context (e.g. fix an urgent bug on `main`) and come back later to resume.

The stash is a stack — you can push multiple stashes and pop them in order.

In [ ]:
# git stash — shelve and restore work in progress
#
#  Stash current changes (tracked files)
#   git stash
#
#  Stash with a descriptive label
#   git stash push -m "WIP: feature engineering for age column"
#
#  Include untracked (new) files in the stash
#   git stash push --include-untracked
#
#  List all stashes
#   git stash list
#   → stash@{0}: WIP: feature engineering for age column
#   → stash@{1}: WIP on main: d1e2f3a Update README
#
#  Apply the most recent stash (keeps it in the stash list)
#   git stash apply
#
#  Apply and remove the most recent stash (most common)
#   git stash pop
#
#  Apply a specific stash by index
#   git stash apply stash@{1}
#
#  Drop a stash you no longer need
#   git stash drop stash@{0}
#
#  Clear all stashes
#   git stash clear

print("git stash pop = apply + remove. Use it when you're done with the interruption.")

---

## 2. Tagging — `git tag`

Tags mark specific commits as significant — usually for releases. Unlike branches, tags do not move when new commits are added.

| Tag type | Description |
|----------|------------|
| **Lightweight** | Just a named pointer — no extra metadata |
| **Annotated** | Stores tagger, date, and a message — preferred for releases |

In [ ]:
# git tag — mark releases
#
#  Create a lightweight tag on the current commit
#   git tag v1.0
#
#  Create an annotated tag (includes message — preferred)
#   git tag -a v1.0 -m "First stable release"
#
#  Tag a specific past commit by its SHA
#   git tag -a v0.9 abc1234 -m "Beta release"
#
#  List all tags
#   git tag
#
#  Show tag details
#   git show v1.0
#
#  Push a single tag to remote
#   git push origin v1.0
#
#  Push all tags
#   git push --tags
#
#  Delete a tag locally
#   git tag -d v1.0
#
#  Delete a tag on the remote
#   git push origin --delete v1.0

print("Annotated tags (-a) store author and message — use them for version releases.")

---

## 3. Rebasing — `git rebase`

`git rebase` moves a branch's commits to a new base commit, rewriting their SHAs in the process. The result is a **clean, linear history** without merge commits.

```
Before rebase:
  main:    A──B──C──E
  feature:       C──D──F

After git rebase main (from the feature branch):
  main:    A──B──C──E
  feature:          E──D'──F'   (D and F are replayed on top of E)
```

> **Golden rule:** Never rebase commits that have already been pushed to a shared remote branch. Rewriting published history causes problems for everyone else.

In [ ]:
# git rebase — common patterns
#
#  Rebase the current feature branch on top of main
#   git switch feature/eda
#   git rebase main
#
#  Interactive rebase — reorder, edit, squash, or drop commits
#   git rebase -i HEAD~3    # interactively edit the last 3 commits
#
#  Interactive rebase options (shown in the editor):
#   pick   → keep commit as-is
#   reword → keep commit, edit the message
#   squash → merge into previous commit
#   fixup  → merge into previous, discard this message
#   drop   → delete the commit
#
#  If a conflict occurs during rebase:
#   1. Resolve the conflict markers in the file
#   2. git add <resolved-file>
#   3. git rebase --continue
#   To abort: git rebase --abort

print("git rebase -i is the most useful form — use it to clean up messy local commits.")

---

## 4. Cherry-picking — `git cherry-pick`

`git cherry-pick` applies the changes from one specific commit onto your current branch. Useful when you want just one fix from a branch without merging the whole branch.

In [ ]:
# git cherry-pick — apply a specific commit
#
#  Apply a single commit by its SHA
#   git cherry-pick abc1234
#
#  Apply a range of commits
#   git cherry-pick abc1234..def5678
#
#  Apply but don't commit yet (stage changes only)
#   git cherry-pick --no-commit abc1234
#
#  Typical use case:
#   You fixed a critical bug on feature/hotfix. You need it on main NOW
#   without merging the whole feature branch.
#
#   git switch main
#   git cherry-pick <sha-of-the-bugfix-commit>

print("cherry-pick copies the diff of a commit — it creates a NEW commit with a new SHA.")

---

## 5. Advanced `git log` — Searching History

Git's log command has powerful filters for finding specific changes across a large history.

In [ ]:
# Advanced git log patterns
#
#  Commits that changed a specific file
#   git log --oneline -- analysis.py
#
#  Commits whose message matches a pattern
#   git log --grep="fix" --oneline
#
#  Commits that added or removed a specific string in the code
#   git log -S "drop_duplicates" --oneline
#
#  Show the actual changes each commit introduced
#   git log -p --follow analysis.py
#
#  Compact graph of all branches
#   git log --oneline --graph --all --decorate
#
#  Show what changed between two commits
#   git diff abc1234 def5678
#
#  Show what changed between two branches
#   git diff main feature/eda

print("git log -S 'string' finds every commit that added or removed that exact string.")

---

## 6. Finding Bugs with `git bisect`

`git bisect` uses binary search to find the exact commit that introduced a bug. You tell it one good commit and one bad commit — it checks out the midpoint and asks you to test. After a few rounds it identifies the culprit.

In [ ]:
# git bisect — binary search for a broken commit
#
# 1. Start bisect
#   git bisect start
#
# 2. Mark the current commit as bad
#   git bisect bad
#
# 3. Mark a known good commit (e.g. a release tag)
#   git bisect good v1.0
#   → Git checks out the midpoint commit
#
# 4. Test the code — if the bug is present, mark bad; if absent, mark good
#   git bisect bad
#   git bisect good
#   (repeat until Git identifies the culprit commit)
#
# 5. Git reports: "abc1234 is the first bad commit"
#
# 6. End bisect (returns to original HEAD)
#   git bisect reset

print("bisect finds the bad commit in log2(N) steps — fast even over thousands of commits.")

---

## 7. The `.gitignore` File

`.gitignore` tells Git which files and folders to never track. It lives in the repo root and is committed like any other file.

**What to ignore:**
- Generated files (`__pycache__/`, `*.pyc`, `.ipynb_checkpoints/`)
- Secrets and credentials (`.env`, `secrets.json`, API keys)
- Large data files (`*.csv`, `*.parquet`, model weights)
- OS and editor artefacts (`.DS_Store`, `.vscode/`)

In [ ]:
# Typical .gitignore for a Python data science project

gitignore_content = """
# Python cache
__pycache__/
*.py[cod]
*.pyo

# Jupyter
.ipynb_checkpoints/

# Virtual environments
.venv/
env/
venv/

# Data files (track with DVC or store in cloud instead)
data/raw/
*.csv
*.parquet
*.h5

# Model weights (often large)
models/
*.pkl
*.joblib

# Secrets — NEVER commit these
.env
secrets.json
credentials.json

# OS files
.DS_Store
Thumbs.db

# IDE settings
.vscode/
.idea/
"""

print(".gitignore entries use glob patterns — ** matches any number of subdirectories.")
print("Use gitignore.io to generate templates for your stack.")

In [ ]:
# Useful .gitignore commands
#
#  Check why a file is (or isn't) being ignored
#   git check-ignore -v filename.csv
#
#  Remove a file that was accidentally committed (stop tracking it)
#   git rm --cached secrets.json
#   # Then add secrets.json to .gitignore and commit
#
#  See what files are currently ignored
#   git status --ignored
#
#  Global .gitignore (applies to all repos on this machine)
#   git config --global core.excludesfile ~/.gitignore_global

print("If you accidentally commit a secret: rotate the key immediately — it is in the history.")

---

## 8. Git in Data Science Projects

Data science repos have unique challenges: large binary files (datasets, model weights) and notebooks that produce noisy diffs. Knowing what to track — and what not to — keeps the repo clean and fast.

| What to track with Git | What NOT to track with Git |
|------------------------|---------------------------|
| Code (`.py`, `.ipynb`) | Large datasets (use DVC or cloud storage) |
| Configuration (`config.yaml`) | Model weights (`.pkl`, `.h5`) |
| `requirements.txt` / `environment.yml` | Rendered outputs (`figures/`, `reports/`) |
| `.gitignore`, `README.md` | Credentials and API keys |
| Tests | `__pycache__/`, `.ipynb_checkpoints/` |

In [ ]:
# Notebook-specific Git tips
#
#  Problem: notebook output cells make diffs unreadable
#  Solution: clear outputs before committing
#
#  In VS Code: Ctrl+Shift+P → 'Clear All Outputs'
#  In Jupyter: Kernel → Restart & Clear Output
#  Via terminal (requires nbconvert):
#   jupyter nbconvert --clear-output --inplace notebook.ipynb
#
#  Problem: dataset files are too large for Git
#  Solution: use DVC (Data Version Control)
#   pip install dvc
#   dvc init
#   dvc add data/raw/sales.csv   # adds to .dvc tracking, not Git
#   git add data/raw/sales.csv.dvc .gitignore
#   git commit -m "Track raw sales data with DVC"
#
#  Consistent environment: always commit requirements
#   pip freeze > requirements.txt
#   git add requirements.txt
#   git commit -m "Pin package versions"

print("Clear notebook outputs before committing to keep diffs readable.")

---

## 9. Git Aliases

Aliases let you create short custom commands for long or frequently used Git commands. They are stored in `~/.gitconfig`.

In [ ]:
# Setting up useful Git aliases
#
#  Single-letter shortcuts
#   git config --global alias.s  'status'
#   git config --global alias.co 'checkout'
#   git config --global alias.br 'branch'
#   git config --global alias.ci 'commit'
#
#  Better log aliases
#   git config --global alias.lg 'log --oneline --graph --all --decorate'
#   git config --global alias.ll 'log --oneline -10'
#
#  Undo last commit, keep changes staged
#   git config --global alias.undo 'reset --soft HEAD~1'
#
#  After setup:
#   git lg     → full graph history
#   git s      → git status
#   git undo   → un-commit the last commit
#
#  View all configured aliases
#   git config --get-regexp alias

print("Aliases are stored in ~/.gitconfig under the [alias] section.")

---

## 10. Undoing Committed Mistakes — Revert vs Reset

This is a summary of the most important distinction in Git history management.

| Command | Changes history? | Safe to use on shared branch? | Use case |
|---------|-----------------|-------------------------------|----------|
| `git revert <sha>` | No — adds a new commit | **Yes** | Undo a published commit |
| `git reset --soft HEAD~1` | Yes — removes commit | Only if not pushed | Redo last commit message |
| `git reset --mixed HEAD~1` | Yes | Only if not pushed | Un-commit + un-stage |
| `git reset --hard HEAD~1` | Yes | **Never** on shared | Throw away last commit entirely |

> **Rule of thumb:** If the commit has been pushed to a shared branch, use `git revert`. If it's only local, `git reset` is fine.

In [ ]:
# Practical undo scenarios
#
#  Scenario 1: Committed to main by mistake (not pushed yet)
#   git reset --soft HEAD~1    # un-commit, keep staged
#   git switch -c feature/fix  # move work to correct branch
#   git commit -m "..."
#
#  Scenario 2: Need to undo a pushed commit without rewriting history
#   git revert abc1234
#   git push
#
#  Scenario 3: Accidentally added sensitive data in the last commit (not pushed)
#   git reset --mixed HEAD~1   # un-commit, keep files in working tree
#   echo ".env" >> .gitignore
#   git add .gitignore
#   git commit -m "Add .gitignore, exclude .env"
#
#  Scenario 4: Restore a single file to what it was at a past commit
#   git restore --source=HEAD~2 analysis.py

print("When in doubt: git revert is always the safe choice on shared branches.")

---

## 11. Mini Exercise — Stash and Rebase

Practise the two most powerful advanced commands.

In [ ]:
# Exercise A: Stash
#
# 1. Start editing a file but don't commit
#   echo "in progress" >> notes.md
#   git status   # shows notes.md as modified
#
# 2. Stash the change
#   git stash push -m "WIP notes"
#   git status   # working tree is clean
#
# 3. Do something else (e.g. fix a typo on main)
#   echo "fix" >> README.md
#   git commit -am "Fix typo in README"
#
# 4. Restore stashed work
#   git stash pop
#   git status   # notes.md is modified again

print("Stash lets you swap context without creating a messy 'WIP' commit.")

In [ ]:
# Exercise B: Interactive rebase — squash messy commits
#
# 1. Make 3 small commits
#   echo "step 1" >> analysis.md && git add . && git commit -m "Step 1"
#   echo "step 2" >> analysis.md && git add . && git commit -m "Step 2"
#   echo "step 3" >> analysis.md && git add . && git commit -m "Finish analysis"
#
# 2. Squash them into one clean commit
#   git rebase -i HEAD~3
#   → In the editor: change 'pick' to 'squash' for the 2nd and 3rd lines
#   → Save and close
#   → In the next editor: write a single clean commit message
#
# 3. Verify
#   git log --oneline
#   → Only one commit for all three changes

print("Interactive rebase lets you curate history before sharing it with others.")

---

## Key Takeaways

- `git stash` shelves uncommitted changes so you can switch tasks and return — `git stash pop` restores them.
- Annotated tags (`git tag -a v1.0 -m "message"`) mark release points permanently and should be pushed explicitly.
- `git rebase` replays commits onto a new base for a clean linear history — **never rebase published commits**.
- Interactive rebase (`git rebase -i HEAD~N`) lets you squash, reorder, or reword the last N commits.
- `git cherry-pick <sha>` applies a single commit's changes to the current branch without merging the whole branch.
- A proper `.gitignore` prevents secrets, large data files, and generated outputs from polluting the repository.
- For data science: track code and config with Git; track large data files with DVC; clear notebook outputs before committing.
- Use `git revert` to safely undo published commits; use `git reset` only on local-only commits.

---

*You have now completed the Git series (50A–50D). Combined with the GitHub platform overview in Section 03D, you have a complete version control workflow for data science projects.*